## Preface

This notebook will be used for developing Automated Synapse Tuning.

Greg Glickert has already done good work to develop location-independent STP and PSC tuning using scipy.optimize

This notebook will extend that application to include location-dependent PSC tuning.

This notebook will attempt to use scipy.optimize.differential_evolution.

## Dependencies

In [1]:
# sudo apt update
# sudo apt install python3-pip
# pip3 install numpy
# pip3 install 
# pip3 install scipy
# pip3 install neuron
# pip3 install tqdm
# pip3 install matplotlib
# pip3 install ipywidgets
# pip3 install pandas

## Example of differential_evolution

In [2]:
import time
import numpy as np
from scipy.optimize import minimize, differential_evolution
import os
import multiprocessing
from multiprocessing import Manager
import sys

# mgr = Manager()
# pid_list = mgr.list()

# # Multimodal test function: Rastrigin
# def rastrigin(x):
#     pid = os.getpid()
#     # print(f"Evaluating in PID {pid}")
#     if pid not in pid_list:
#         pid_list.append(pid)
#     A = 10
#     return A * len(x) + sum(xi**2 - A * np.cos(2 * np.pi * xi) for xi in x)

# def show_pool(x, convergence):
#     kids = multiprocessing.active_children()
#     print("Active children:", [p.name for p in kids])
#     return False  # don’t halt

# # Problem setup
# dim    = 4
# bounds = [(-5.12, 5.12)] * dim
# x0     = np.zeros(dim)  # starting guess at the known global minimum

# # 1) Local solver (BFGS) from x0
# t0 = time.perf_counter()
# res_local = minimize(rastrigin, x0, method='BFGS')
# t_local = time.perf_counter() - t0

# # 2) Global solver (Differential Evolution)
# t1 = time.perf_counter()
# res_global = differential_evolution(
#     rastrigin,
#     bounds,
#     workers=4,            # parallel evaluation of the population
#     updating='deferred',   # allows batching of function calls
#     disp=True,
#     # callback=show_pool # show the pool using a callback function
# )
# t_global = time.perf_counter() - t1

# # Report
# print("Worker PIDs:", list(pid_list))
# print("Count of distinct workers:", len(pid_list))

# print(f"Local solver:")
# print(f"  x = {res_local.x}")
# print(f"  fun = {res_local.fun:.6e}")
# print(f"  time = {t_local:.4f} s\n")

# print(f"Global solver:")
# print(f"  x = {res_global.x}")
# print(f"  fun = {res_global.fun:.6e}")
# print(f"  time = {t_global:.4f} s")


## Tuning location-dependent PSCs.

In [3]:
import neuron
from neuron import h

--No graphics will be displayed.


In [4]:
# clone the forked repo for bmtool/synapse_tuner
!git clone https://github.com/davidfague/bmtool.git

fatal: destination path 'bmtool' already exists and is not an empty directory.


In [5]:
# move the modfiles from the other repo
# https://github.com/davidfague/Neural-Modeling/tree/cleanup2/modfiles/hay -> /content/bmtool/examples/synapses

# !mv Neural-Modeling/modfiles/hay/* /content/bmtool/examples/synapses/modfiles/
!mv ../modfiles/hay/* bmtool/examples/synapses/modfiles/

mv: cannot stat '../modfiles/hay/*': No such file or directory


In [6]:
# import os
# os.chdir('/content/bmtool/examples/synapses')
os.chdir('bmtool/examples/synapses')

In [7]:
# import sys
# sys.path.append('../../bmtool/')
sys.path.append('../../../bmtool/')

In [8]:
import bmtool

In [9]:
# having 2 h.vecstim objects seems to give an error & removing one of these seemed to fix
# !rm modfiles/vecevent_coreneuron.mod # keep this one assuming it is faster or more optimized or a later version that encompasses the other
!rm modfiles/vecevent.mod

rm: cannot remove 'modfiles/vecevent.mod': No such file or directory


In [10]:
# if already compiled then lets delete the folder and force a recompile
if os.path.isdir('modfiles/x86_64'):
    os.system("rm -rf modfiles/x86_64 ")
# compile the mod files
if not os.path.isdir("modfiles/x86_64"):
    os.chdir('modfiles')
    os.system("nrnivmodl > /dev/null 2>&1") # hide output
    # print(os.system("nrnivmodl")) # show output
    os.chdir("..")

In [11]:
# !nrnivmodl modfiles/ # show output
!nrnivmodl modfiles/ > /dev/null 2>&1 # hide output

In [12]:
ls ../../../../Modules/

adjacency.py                 logger.py                  segments_file.py
allen_interfacing.py         morphology_manipulator.py  simulation.py
analysis.py                  morph_reduction_utils.py   simulation_slurm.py
cable_expander_func.py       notebook_utils.py          spike_generator.py
cell_builder.py              plot_morphology.py         stylized_builder.py
cell_model.py                plot_voltage.py            stylized_module.py
constants.py                 presynaptic_old.py         surface_area.py
ecp.py                       presynaptic.py             synapse.py
electrotonic_distance.py     __pycache__/               synapses_file.py
event_histograms.py          recorder.py                view_synapses.py
general_settings_for_AST.py  reduction.py
group_simulations.py         reduction_utils.py


In [13]:
# os.path.append("../../../../Modules/general_settings_for_AST")
sys.path.append("../../../../Modules/")
from general_settings_for_AST import general_settings, conn_type_settings

## replacing the FSi_sec

In [14]:
# sys.path.append("/content/bmtool/bmtool/")
sys.path.append("/content/bmtool/bmtool/util/synapses.py")

In [15]:
neuron.load_mechanisms('modfiles/') # have to load the mechanisms to load L5PCbiophys3.hoc

True

In [16]:
use_hay_cell = True # overwrite the target cell
from general_settings_for_AST import load_hay_cell
if use_hay_cell:
    template_arg = load_hay_cell(conn_type_settings)
else:
    template_arg=None
    hoc_files_to_load = None

In [17]:
# have to unload the mechanisms or the synapse tuner throws an error
!rm -rf x86_64
!rm -rf modfiles/x86_64

In [18]:
sys.path.append("/content/bmtool/bmtool/")

In [19]:
from general_settings_for_AST import tuner_configs, InitializeSysnapseTuner

ben_synapses = True  # or False
selected = tuner_configs[ben_synapses]

# instantiate all tuners in one line:
# tuners = {
#     name: InitializeSysnapseTuner(template_arg=template_arg, **params)
#     for name, params in selected.items()
# }

In [20]:
selected.keys()

dict_keys(['exc', 'inhPerisomatic', 'inhDendritic'])

In [21]:
from general_settings_for_AST import target_metrics, location_types_by_synapse_type, distributions_to_test

In [22]:
# location_types_by_synapse_type

In [37]:
# simulation code for gather PSCs across locations and weights.
import multiprocessing as mp
import datetime
from general_settings_for_AST import log_norm_dist, norm_dist
total_samples_per_weight_distribution = 100
# simulation_batch_size=2 # number of simulations to run in parallel
location_type = ''
segments_of_loc_type = []
synapse_type = 'exc'
location_type = 'distal_basal'
use_norm_dist = False

exc_mean = (np.log(0.45) - 0.5 * np.log((0.35/0.45)**2+1))
exc_std = np.sqrt(np.log((0.35/0.45)**2 + 1))
exc_clip = (1e-15,5)

cpu_cores = mp.cpu_count() # detect how many cores are available

# choose  batch size = number of workers used
# (or cpu_cores-1 to leave one core free for the OS)
simulation_batch_size = cpu_cores - 1

# to make sure you never ask for more sims than you actually need:
simulation_batch_size = min(total_samples_per_weight_distribution, simulation_batch_size)


# round total_samples_per_weight_distributions up so that there is an integar number of batches for (total_samples_per_weight_distribution / simulation_batch_size)
number_of_batches = int(total_samples_per_weight_distribution / simulation_batch_size)

def get_sec_ids_from_type(section_type):
  cell = h.L5PCtemplate("../../../../cells/templates/cell1.asc")

  if section_type == 'distal_apic': # distal apic (>100 microns from soma)
    sec_ids_to_use = [idx for idx,sec in enumerate(cell.all) if (sec in cell.apic) and (h.distance(cell.soma[0](0.5), sec(0.5)) > 100)]
  elif section_type == 'distal_basal': # distal basal dendrites (>100 microns from soma)
    sec_ids_to_use = [idx for idx,sec in enumerate(cell.all) if (sec in cell.dend) and (h.distance(cell.soma[0](0.5), sec(0.5)) > 100)]
  elif section_type == 'perisomatic': # proximal dendrites and soma  (within 100 microns of soma)
    sec_ids_to_use = [idx for idx,sec in enumerate(cell.all) if ((h.distance(cell.soma[0](0.5), sec(0.5)) < 100) and (sec not in list(cell.axon)))]
  else:
    del cell
    NotImplementedError(f"{section_type} not implemented for get_sec_ids_from_type")

  del cell
  return sec_ids_to_use

def get_segments(synapse_tuner_obj: object, location_type: str):
    segments = [seg for sec in synapse_tuner_obj.cell.all for seg in sec]
    sec_ids_to_use = get_sec_ids_from_type(location_type) # select the type using integar
    possible_segments = [seg for sec_id in sec_ids_to_use for seg in list(synapse_tuner_obj.cell.all)[sec_id]]
    seg_probs = [(seg.sec.L / seg.sec.nseg) for seg in possible_segments]
    return segments, possible_segments, seg_probs

def move_synapse_to_new_location(synapse_tuner_obj: object, possible_segments, seg_probs, random_state, all_segments):
    seg_to_place_syn_on = np.random.choice(possible_segments, 1, True, seg_probs / np.sum(seg_probs))[0] # random_state.choice(possible_segments, 1, True, seg_probs / np.sum(seg_probs))[0]
    synapse_tuner_obj.syn.loc(seg_to_place_syn_on) # move the synapse to the target location
    segment_index = all_segments.index(seg_to_place_syn_on)
    return segment_index

def change_synapse_weight(synapse_tuner_obj: object, distributions_to_test, synapse_type, location_type, use_norm_dist):
    if use_norm_dist:
        new_weight = norm_dist(distributions_to_test[synapse_type][location_type]['mean'], distributions_to_test[synapse_type][location_type]['std'], 1, (0, 10*distributions_to_test[synapse_type][location_type]['mean']))
    else:
        new_weight = log_norm_dist(exc_mean, exc_std, 1, exc_clip, distributions_to_test[synapse_type][location_type]['exc_scalar'])
    synapse_tuner_obj.syn.initW = new_weight
    return new_weight

def simulate_PSC(synapse_type, tuner_configs, location_type: str, distributions_to_test, use_norm_dist: bool):
    synapse_tuner_obj =  InitializeSysnapseTuner(template_arg=template_arg, **tuner_configs[True][synapse_type])#STOPPING POINT
    all_segments, possible_segments, seg_probs = get_segments(synapse_tuner_obj, location_type)
    weight = change_synapse_weight(synapse_tuner_obj, distributions_to_test, 'exc' if 'exc' in synapse_type.lower() else 'inh',
                                   location_type, use_norm_dist)
    loc = move_synapse_to_new_location(synapse_tuner_obj, possible_segments, seg_probs, None, all_segments)
    PSC_mag = max(abs(synapse_tuner_obj.SingleEvent(plot_and_print=False))) # NOTE: have to update to return
    return PSC_mag, weight, loc

PSC_mags = []
weights = []
locs = []
start = time.time()
# # run {simulation_batch_size} simulations for {number_of_batches}, recording PSC for each simulation and location of synapse.
for batch_idx in range(number_of_batches):
    for simulation_idx in range(simulation_batch_size): # parallelize this part
        PSC_mag, weight, loc = simulate_PSC(synapse_type, tuner_configs, location_type, distributions_to_test, use_norm_dist)
        PSC_mags.append(PSC_mag)
        weights.append(weight)
        locs.append(loc)
print(f"elapsed time: {time.time() - start}")

PSC_mags = []
weights = []
locs = []
start = time.time()
# create a pool with as many workers as sims we want in parallel
with mp.Pool(processes=simulation_batch_size) as pool:
    for _ in range(number_of_batches):
        # build the list of arguments for this batch
        batch_args = [
            (synapse_type, tuner_configs, location_type, distributions_to_test, use_norm_dist)
            for _ in range(simulation_batch_size)
        ]
        # run them in parallel and collect results
        results = pool.starmap(simulate_PSC, batch_args)
        # unpack into your master lists
        for PSC_mag, weight, loc in results:
            PSC_mags.append(PSC_mag)
            weights.append(weight)
            locs.append(loc)
print(f"elapsed time: {time.time() - start}")

print(PSC_mags)
# compute error

# iteratively adjust input_weight_distribution according to error.

NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanism

In [ ]:
import multiprocessing as mp
import numpy as np
import pandas as pd
import time
from math import ceil
from general_settings_for_AST import log_norm_dist, norm_dist
# from your_module import InitializeSysnapseTuner, get_sec_ids_from_type  # adjust imports

# ─── User‐defined globals ──────────────────────────────────────────────────────
numpy_random_state = 4277176
random_state = np.random.RandomState(numpy_random_state)
num_weights_per_loc = 2
exc_mean = (np.log(0.45) - 0.5 * np.log((0.35/0.45)**2 + 1))
exc_std  = np.sqrt(np.log((0.35/0.45)**2 + 1))
exc_clip = (0, 5)

# These must already be defined in your namespace:
#   synapses
#   location_types_by_synapse_type
#   tuner_configs
#   distributions_to_test
#   target_metrics
#   template_arg

# Prepare data structures
results = []
resulting_PSCs_by_segment = {}
tuners = {
    name: InitializeSysnapseTuner(template_arg=template_arg, **params)
    for name, params in selected.items()
}
for syn in location_types_by_synapse_type:
    # flat list of all segments on this cell
    all_segs = [seg for sec in tuners[syn].cell.all for seg in sec]
    resulting_PSCs_by_synapse = {'segments': all_segs}
    # one empty list-of-lists per segment for each location_type
    for loc in location_types_by_synapse_type[syn]:
        resulting_PSCs_by_synapse[loc] = [[] for _ in all_segs]
    resulting_PSCs_by_segment[syn] = resulting_PSCs_by_synapse

# ─── Helper to run one PSC draw ────────────────────────────────────────────────
def simulate_single_psc(args):
    synapse_type, location_type = args
    # instantiate fresh tuner
    tuner = InitializeSysnapseTuner(template_arg=template_arg,
                                   **tuner_configs[True][synapse_type])
    # avoid stale sliders
    if hasattr(tuner, 'dynamic_sliders'):
        del tuner.dynamic_sliders

    # all segments & the ones for this loc
    all_segs = [seg for sec in tuner.cell.all for seg in sec]
    sec_ids = get_sec_ids_from_type(location_type)
    # print(tuner.cell.all)
    possible_segs = [seg for sid in sec_ids for seg in list(tuner.cell.all)[sid]]
    probs = np.array([seg.sec.L/seg.sec.nseg for seg in possible_segs], dtype=float)
    probs /= probs.sum()

    # pick segment & move synapse
    seg = random_state.choice(possible_segs, p=probs)
    tuner.syn.loc(seg)
    seg_idx = all_segs.index(seg)

    # pick weight
    if 'exc' in synapse_type:
        w = log_norm_dist(exc_mean, exc_std, 1, exc_clip,
                          distributions_to_test[synapse_type][location_type]['exc_scalar'])
    else:
        w = norm_dist(distributions_to_test[synapse_type][location_type]['mean'],
                      distributions_to_test[synapse_type][location_type]['std'],
                      1,
                      (0, 10*distributions_to_test[synapse_type][location_type]['mean']))
    tuner.syn.initW = w

    # record PSC magnitude
    psc = max(abs(tuner.SingleEvent(plot_and_print=False)))
    return synapse_type, location_type, seg_idx, psc

# ─── Main loop: for each synapse_type & location_type ──────────────────────────
for synapse_type, loc_list in location_types_by_synapse_type.items():
    for location_type in loc_list:
        # figure out how many draws we need
        # (same as your serial: at least segments*weights or 200)
        sec_ids = get_sec_ids_from_type(location_type)
        # n_segs = sum(len(tuner.cell.all[sid]) for sid in sec_ids
        #              for tuner in [synapses[synapse_type]])
        n_tests = 10000#max(n_segs * num_weights_per_loc, 200)

        # build argument list: one entry per PSC draw
        task_args = [(synapse_type, location_type)] * n_tests

        # run in parallel
        pool_size = max(mp.cpu_count() - 1, 1)
        start = time.time()
        with mp.Pool(pool_size) as pool:
            all_results = pool.map(simulate_single_psc, task_args)
        elapsed = time.time() - start
        print(f"{synapse_type} @ {location_type}: ran {n_tests} sims in {elapsed:.1f}s")

        # unpack results
        mags = []
        for syn, loc, seg_idx, psc in all_results:
            mags.append(psc)
            resulting_PSCs_by_segment[syn][loc][seg_idx].append(psc)

        # compute stats + errors
        mean_psc = np.mean(mags)
        std_psc  = np.std(mags)
        tgt = target_metrics[synapse_type]['max_amplitude']
        mean_err = tgt['mean'] - mean_psc
        std_err  = tgt['std']  - std_psc

        # log & store
        print(f"  target: mean={tgt['mean']:.2f}, std={tgt['std']:.2f}")
        print(f"  actual: mean={mean_psc:.2f}, std={std_psc:.2f}")
        print(f"  error:  mean={mean_err:.2f}, std={std_err:.2f}\n")

        results.append({
            "Synapse Type": synapse_type,
            "Location Type": location_type,
            "initW_mean": round(distributions_to_test[synapse_type][location_type]['mean'], 3),
            "initW_std":  round(distributions_to_test[synapse_type][location_type]['std'], 3),
            "PSC Mean":    round(mean_psc, 3),
            "PSC Std":     round(std_psc, 3),
            "PSC_mean_error": round(mean_err, 3),
            "PSC_std_error":  round(std_err, 3),
            "n_tests": n_tests,
        })

# assemble into DataFrame
results_df = pd.DataFrame(results)
# display or export as before
# import ace_tools as tools
# tools.display_dataframe_to_user(name="PSC Data", dataframe=results_df)


NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.


NEURON mechanisms not found in modfiles.
NEURON mechanisms not found

In [26]:
import numpy as np
from scipy.optimize import minimize

def draw_one_psc(synapse_type: str,
                 location_type: str,
                 rng: np.random.RandomState) -> float:
    """
    Draws a single PSC magnitude by:
      1. Instantiating a fresh tuner
      2. Randomly placing it on one segment (weighted by segment length)
      3. Sampling one synaptic weight from the current distributions_to_test
      4. Recording the SingleEvent PSC magnitude

    Returns
    -------
    psc_mag : float
        max(abs(PSC)) from tuner.SingleEvent(...)
    """
    # 1) instantiate
    tuner = InitializeSysnapseTuner(
        template_arg=template_arg,
        **tuner_configs[True][synapse_type]
    )
    if hasattr(tuner, "dynamic_sliders"):
        del tuner.dynamic_sliders

    # 2) build segment list + probabilities
    all_segs = [seg for sec in tuner.cell.all for seg in sec]
    sec_ids   = get_sec_ids_from_type(location_type)
    poss_segs = [seg for sid in sec_ids for seg in list(tuner.cell.all)[sid]]
    probs     = np.array([seg.sec.L/seg.sec.nseg for seg in poss_segs], dtype=float)
    probs    /= probs.sum()

    # choose & move
    chosen_seg = rng.choice(poss_segs, p=probs)
    tuner.syn.loc(chosen_seg)

    # 3) sample weight from your (possibly just-updated) distributions_to_test
    mean_w = distributions_to_test[synapse_type][location_type]["mean"]
    std_w  = distributions_to_test[synapse_type][location_type]["std"]

    if "exc" in synapse_type:
        # convert to log‐normal params (μ_log, σ_log)
        mu_log    = np.log(mean_w**2/np.sqrt(std_w**2 + mean_w**2))
        sigma_log = np.sqrt(np.log(1 + (std_w**2)/(mean_w**2)))
        scalar    = distributions_to_test[synapse_type][location_type]["exc_scalar"]
        w = log_norm_dist(mu_log, sigma_log, 1, exc_clip, scalar)[0]
    else:
        # inhibitory: simple normal clipped at [0,10*mean]
        w = norm_dist(mean_w, std_w, 1, (0, 10*mean_w))#[0]

    tuner.syn.initW = w

    # 4) record and return
    psc = tuner.SingleEvent(plot_and_print=False)
    return float(np.max(np.abs(psc)))


# ─── 1. Objective function ────────────────────────────────────────────────────
def weight_dist_error(params, synapse_type, location_type,
                      n_eval=200,  # fewer sims for speed during fitting
                      random_seed=1234):
    """
    params: [mean, std] for the weight distribution
    returns: sum of squared errors between (simulated_psc_mean, simulated_psc_std)
             and (target_metrics['mean'], target_metrics['std'])
    """
    mean_w, std_w = params
    # enforce positivity
    if mean_w <= 0 or std_w <= 0:
        return np.inf

    # patch your distribution
    distributions_to_test[synapse_type][location_type]['mean'] = mean_w
    distributions_to_test[synapse_type][location_type]['std']  = std_w

    # rerun a short simulation batch
    rng = np.random.RandomState(random_seed)
    mags = []
    for _ in range(n_eval):
        # mirror exactly your simulate_single_psc but only vary the weights
        # 1) sample a segment & move synapse (reuse your code)
        # 2) draw w = log_norm_dist(mean_w, std_w, ...)
        # 3) record psc

        # … for brevity, assume you have a function:
        #    psc = draw_one_psc(synapse_type, location_type, rng)
        # which internally uses mean_w, std_w from distributions_to_test.
        psc = draw_one_psc(synapse_type, location_type, rng)
        mags.append(psc)

    sim_mean = np.mean(mags)
    sim_std  = np.std(mags)
    tgt      = target_metrics[synapse_type]['max_amplitude']

    # sum of squared errors
    err = (sim_mean - tgt['mean'])**2 + (sim_std - tgt['std'])**2
    return err


# ─── 2. Run optimization for every (synapse,location) ──────────────────────────
optimized_params = {}

for syn in location_types_by_synapse_type:
    optimized_params[syn] = {}
    for loc in location_types_by_synapse_type[syn]:
        # initial guess from your existing distributions
        init_mean = distributions_to_test[syn][loc]['mean']
        init_std  = distributions_to_test[syn][loc]['std']

        # bounds: mean>0, std>0
        bnds = [(1e-6, None), (1e-6, None)]

        res = minimize(
            weight_dist_error,
            x0=[init_mean, init_std],
            args=(syn, loc, 200, 4277176),
            method='L-BFGS-B',
            bounds=bnds,
            options={'maxiter': 50, 'disp': True}
        )

        print(f"⇨ {syn}@{loc} → success={res.success}, error={res.fun:.3f}")
        print(f"    best mean={res.x[0]:.4f}, std={res.x[1]:.4f}\n")

        # save them
        optimized_params[syn][loc] = {'mean': res.x[0], 'std': res.x[1]}

# now optimized_params holds your fitted (mean,std) for each synapse_type & location.


NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.


KeyboardInterrupt: 

In [ ]:
import numpy as np
from scipy.optimize import minimize
import multiprocessing as mp

numpy_random_state =1244152

# ─── A) Seeded sampler ────────────────────────────────────────────────────────
def draw_one_psc_seeded(synapse_type: str,
                        location_type: str,
                        seed: int) -> float:
    """
    Exactly like your draw_one_psc, but uses its own RandomState(seed).
    """
    rng = np.random.RandomState(seed)

    # 1) instantiate tuner
    tuner = InitializeSysnapseTuner(
        template_arg=template_arg,
        **tuner_configs[True][synapse_type]
    )
    if hasattr(tuner, "dynamic_sliders"):
        del tuner.dynamic_sliders

    # 2) pick segment
    all_segs = [seg for sec in tuner.cell.all for seg in sec]
    sec_ids   = get_sec_ids_from_type(location_type)
    poss_segs = [seg for sid in sec_ids for seg in list(tuner.cell.all)[sid]]
    probs     = np.array([seg.sec.L/seg.sec.nseg for seg in poss_segs], float)
    probs    /= probs.sum()
    chosen   = rng.choice(poss_segs, p=probs)
    tuner.syn.loc(chosen)

    # 3) sample weight from current distributions_to_test
    mean_w = distributions_to_test[synapse_type][location_type]["mean"]
    std_w  = distributions_to_test[synapse_type][location_type]["std"]

    if "exc" in synapse_type:
        mu_log    = np.log(mean_w**2/np.sqrt(std_w**2 + mean_w**2))
        sigma_log = np.sqrt(np.log(1 + (std_w**2)/(mean_w**2)))
        scalar    = distributions_to_test[synapse_type][location_type]["exc_scalar"]
        w = log_norm_dist(mu_log, sigma_log, 1, exc_clip, scalar)#[0]
    else:
        w = norm_dist(mean_w, std_w, 1, (0, 10*mean_w))#[0]

    tuner.syn.initW = w

    # 4) record PSC magnitude
    psc = tuner.SingleEvent(plot_and_print=False)
    return float(np.max(np.abs(psc)))


# ─── B) Parallel objective ───────────────────────────────────────────────────
def weight_dist_error(params,
                      synapse_type, location_type,
                      n_eval=200, base_seed=1234):
    mean_w, std_w = params
    if mean_w <= 0 or std_w <= 0:
        return np.inf

    # update distribution
    distributions_to_test[synapse_type][location_type]["mean"] = mean_w
    distributions_to_test[synapse_type][location_type]["std"]  = std_w

    # build seeds & argument list
    seeds = np.random.RandomState(base_seed).randint(0, 2**31-1, size=n_eval)
    args  = [(synapse_type, location_type, int(s)) for s in seeds]

    # run them in parallel
    pool_size = max(mp.cpu_count() - 1, 1)
    with mp.Pool(pool_size) as pool:
        mags = pool.starmap(draw_one_psc_seeded, args)

    sim_mean = np.mean(mags)
    sim_std  = np.std(mags)
    tgt      = target_metrics[synapse_type]["magnitude"]

    return (sim_mean - tgt["mean"])**2 + (sim_std - tgt["std"])**2


# ─── C) Wrap your optimize loop under the guard ────────────────────────────────
if __name__ == "__main__":
    optimized_params = {}
    for syn in location_types_by_synapse_type:
        optimized_params[syn] = {}
        for loc in location_types_by_synapse_type[syn]:
            init_mean = distributions_to_test[syn][loc]["mean"]
            init_std  = distributions_to_test[syn][loc]["std"]
            bnds = [(1e-6, None), (1e-6, None)]

            res = minimize(
                weight_dist_error,
                x0=[init_mean, init_std],
                args=(syn, loc, 200, numpy_random_state),
                method="L-BFGS-B",
                bounds=bnds,
                options={"maxiter": 50, "disp": True},
            )

            print(f"{syn}@{loc}: success={res.success}, "
                  f"mean={res.x[0]:.4f}, std={res.x[1]:.4f}")
            optimized_params[syn][loc] = {"mean": res.x[0], "std": res.x[1]}

    # optimized_params now holds your fitted values.


NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.

NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.

NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.

NEURON mechanisms not found in modfiles.NEURON mechanisms not found in modfiles.






NEURON mechanisms not found in modfiles.



NEURON mechanisms not found in modfiles.
NEURON mechanisms not found in modfiles.NEURON mechanisms not

In [ ]:
# tuners

In [ ]:
# location_types_by_synapse_type

gather the post synaptic current magnitude across the regions

In [ ]:
# gather PSC across different synaptic weights (weights only for exc; inh are fixed) & regions
measure_PSCs = True

num_weights_per_loc = 2 # number of tests per segment (although each test will have synapse move to random seg with probability seg_length)

numpy_random_state = 4277176
random_state = np.random.RandomState(numpy_random_state)

exc_mean = (np.log(0.45) - 0.5 * np.log((0.35/0.45)**2+1))
exc_std = np.sqrt(np.log((0.35/0.45)**2 + 1))
exc_clip = (0,5)

# inhDendritic:(0.16842415731075505, 0.08474158821873527)
# Initialize an empty list to store results
segments = [seg for sec in synapses['exc'].cell.all for seg in sec]
results = []
resulting_PSCs_by_segment = {}
for synapse_type in location_types_by_synapse_type.keys():
  resulting_PSCs_by_segment[synapse_type] = {} # make 1 for each tuner/synapse type
  resulting_PSCs_by_segment[synapse_type]['segments'] = [seg for sec in synapses[synapse_type].cell.all for seg in sec] # list segments for each tuner Segments are the same, but identified differently bc each tuner built its own cell.
  for location_type in location_types_by_synapse_type[synapse_type]: # make one for each location type
    resulting_PSCs_by_segment[synapse_type][location_type] = [[] for seg in resulting_PSCs_by_segment[synapse_type]['segments']] # initialize list of PSCs that were used for each segment

if measure_PSCs:
  # gather distribution of PSC magnitudes for each synapse_type and location types
  for synapse_type, location_types in location_types_by_synapse_type.items():
    tuner = synapses[synapse_type]
    if hasattr(synapses[synapse_type], 'dynamic_sliders'):
      del synapses[synapse_type].dynamic_sliders # this var would replace the values even when we try to update them
    if 'inh' in synapse_type:
      num_weights_per_loc_to_use = num_weights_per_loc
      resample_weight = True
      use_norm_dist = True
    else:
      num_weights_per_loc_to_use = num_weights_per_loc
      resample_weight = True
      use_norm_dist = False

    for location_type in location_types:
      # get the sections we need
      sec_ids_to_use = get_sec_ids_from_type(location_type) # select the type using integar

      if synapse_type == 'exc':
        exc_gmax_scalar = distributions_to_test[synapse_type][location_type]['exc_scalar']

      # gather PSC magnitudes across locations
      magnitudes = []

      possible_segments = [seg for sec_id in sec_ids_to_use for seg in list(tuner.cell.all)[sec_id]]
      seg_probs = [(seg.sec.L / seg.sec.nseg) for seg in possible_segments]

      n_tests = max(len(possible_segments)*num_weights_per_loc, 200)

      for i in range(n_tests):
        # randomly pick segment based on probabilities
        seg_to_place_syn_on = random_state.choice(possible_segments, 1, True, seg_probs / np.sum(seg_probs))[0]

        # add PSC result for this segment
        segment_index = resulting_PSCs_by_segment[synapse_type]['segments'].index(seg_to_place_syn_on)

        # move the synapse to the target location
        tuner.syn.loc(seg_to_place_syn_on)

        for i_weight in range(num_weights_per_loc_to_use):
          if resample_weight:
            if use_norm_dist:
              new_weight = norm_dist(distributions_to_test[synapse_type][location_type]['mean'], distributions_to_test[synapse_type][location_type]['std'], 1, (0, 10*distributions_to_test[synapse_type][location_type]['mean']))
            else:
              new_weight = log_norm_dist(exc_mean, exc_std, 1, exc_clip, exc_gmax_scalar)
            tuner.syn.initW = new_weight

          #record magnitude of PSC
          PSC_mag = max(abs(tuner.SingleEvent(plot_and_print=False))) # NOTE: have to update to return
          magnitudes.append(PSC_mag) # have to fix this line
          resulting_PSCs_by_segment[synapse_type][location_type][segment_index].append(PSC_mag) # track by segment

      # print(f"distributions_to_test {synapse_type}: {distributions_to_test[synapse_type]")

      # calc mean, std
      psc_mean = np.mean(magnitudes)
      psc_std = np.std(magnitudes)
      print(f"{synapse_type} {location_type}")
      print(f" target_metrics: {target_metrics[synapse_type]}")
      print(f" actual: mean:{psc_mean:.2f}, std:{psc_std:.2f}")

      #show error
      print(f" error: mean:{target_metrics[synapse_type]['max_amplitude']['mean'] - psc_mean:.2f} std:{target_metrics[synapse_type]['max_amplitude']['std'] - psc_std:.2f}\n")

      # Store results in the list
      results.append({
          "Synapse Type": (synapse_type),
          "Location Type": location_type,
          "initW_mean": round(distributions_to_test[synapse_type][location_type]['mean'], 3),
          "initW_std": round(distributions_to_test[synapse_type][location_type]['std'], 3),
          "PSC Mean": round(psc_mean, 3),
          "PSC Std": round(psc_std, 3),
          "PSC_mean_error": round(target_metrics[synapse_type]['max_amplitude']['mean'] - psc_mean, 3),
          "PSC_std_error": round(target_metrics[synapse_type]['max_amplitude']['std'] - psc_std, 3),
          "n_tests": n_tests,
      })

# Convert the list to a DataFrame
results_df = pd.DataFrame(results)
# import ace_tools as tools
# tools.display_dataframe_to_user(name="PSC Data", dataframe=results_df)

In [ ]:
import pickle

# Save to a file
with open('resulting_PSCs_by_segment.pkl', 'wb') as f:
    pickle.dump(resulting_PSCs_by_segment, f)

# Load from a file
with open('resulting_PSCs_by_segment.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

In [ ]:
for synapse_type,psc_data in resulting_PSCs_by_segment.items():
  resulting_PSCs_by_segment[synapse_type]['segments'] = str(psc_data['segments'])

In [ ]:
# desired table:
PSC_mean_array = []
PSC_std_array = []

for syn_type, loc_types in location_types_by_synapse_type.items():
  for loc_type in loc_types:
    PSC_mean_array.append(target_metrics[syn_type]['max_amplitude']['mean'])
    PSC_std_array.append(target_metrics[syn_type]['max_amplitude']['std'])

expected_data = {
    'Synapse Type': ['inhPerisomatic', 'exc', 'exc', 'inhDendritic', 'inhDendritic'],
    'Location Type': ['perisomatic', 'distal_basal', 'distal_apic', 'distal_basal', 'distal_apic'],
    'PSC Mean': PSC_mean_array,
    'PSC Std': PSC_std_array
}

# Create the DataFrame
expected_results_df = pd.DataFrame(expected_data)

# Display the DataFrame (optional)
expected_results_df